# 06 · Iterative self-repair on a real test suite

How to turn a one-shot solver into an agent that **iterates
against a real `pytest` suite** until it's green.

**Key concepts**
- **Executable feedback** — instead of a single guess, the model reads actual test
  failures and revises, mimicking how a developer debugs.
- **Test suite as reward** — the pytest exit code / output is the objective signal;
  the loop ends when the suite passes or the attempt budget is exhausted.
- Builds directly on `02` (feedback loop) but with a *real* multi-test suite rather
  than a single check.

**How it works:** write `solution.py` + `test_solution.py`, run pytest, feed the
tail of the failure output back to the model, repeat.

**Prerequisites:** AWS credentials + Bedrock model access. (`agentcore-ci` has pytest.)

In [ ]:
%pip install -q raitbox

### Configuration & task
Bedrock client, the `SPEC` the model must implement, and the pytest `TESTS` it's graded against.

In [ ]:
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

SYSTEM_PROMPT = [{"text": (
    "You implement the requested Python module. Reply with ONLY the module code "
    "(no prose, no markdown fences, no tests). When given a failing traceback, "
    "return the full corrected module that fixes it."
)}]

SPEC = (
    "Implement `def roman_to_int(s: str) -> int` converting a Roman numeral string "
    "to an integer (support subtractive forms like IV, IX, XL, CM)."
)
TESTS = "\n".join([
    "import pytest",
    "@pytest.mark.parametrize('s,n', [('III',3),('IV',4),('IX',9),('LVIII',58),('MCMXCIV',1994)])",
    "def test_roman(s, n):",
    "    assert roman_to_int(s) == n",
])

### The repair loop
Write the module + tests, run pytest, and on failure feed the traceback tail back to the model. `CommandExitException` is caught so a failing suite is *feedback*, not a crash.

In [ ]:
from raitbox import CommandExitException


def ask_model(messages) -> str:
    resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM_PROMPT, messages=messages,
                            inferenceConfig={"maxTokens": 1500, "temperature": 0})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (fenced[0] if fenced else text).strip()


def self_repair(spec, tests, max_attempts=4):
    messages = [{"role": "user", "content": [{"text": spec}]}]
    with Sandbox.create(region_name=REGION) as sbx:
        module = ""
        for attempt in range(1, max_attempts + 1):
            module = ask_model(messages)
            sbx.files.write("solution.py", module)
            sbx.files.write("test_solution.py", "from solution import *\n" + tests)
            # pytest exits non-zero when tests fail, which raises CommandExitException;
            # catch it and treat the failure output as feedback rather than an error.
            try:
                res = sbx.commands.run("python3 -m pytest -q test_solution.py", timeout=120)
                out, failed = res.stdout, False
            except CommandExitException as e:
                out, failed = e.stdout, True
            print(f"--- attempt {attempt}: {'FAIL' if failed else 'PASS'} ---")
            if not failed:
                return True, attempt, module
            tail = "\n".join(out.splitlines()[-25:])
            print(tail)
            messages.append({"role": "assistant", "content": [{"text": module}]})
            messages.append({"role": "user", "content": [{"text":
                "The tests failed:\n" + tail + "\nReturn the full corrected module."}]})
    return False, max_attempts, module

## Run the repair loop

(`agentcore-ci` has pytest; the harness writes `solution.py` + `test_solution.py` and runs the suite.)

In [ ]:
ok, attempts, final_module = self_repair(SPEC, TESTS)
print(f"\n{'SOLVED' if ok else 'UNSOLVED'} in {attempts} attempt(s)")
print("\nfinal module:\n" + final_module)

## Recap & where to go next
You built a debugging agent that fixes its code until a real test suite passes.

**Related:** `02` (simpler loop) · `14` (patch-based, git) · `15` (RL-env framing).